<a href="https://colab.research.google.com/github/madhan112007/Emotion-analysis-and-Object-Detection/blob/main/Emotion_analysis_and_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import gradio as gr
import cv2
from deepface import DeepFace
from ultralytics import YOLO
import numpy as np
from PIL import Image


yolo_model = YOLO("yolov8n.pt")


def analyze_image(image):
    img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    objects = []
    emotions = []


    results = yolo_model.predict(img_rgb, conf=0.3)[0]
    for box in results.boxes.data:
        x1, y1, x2, y2, conf, cls = box.cpu().numpy()
        label = results.names[int(cls)]
        objects.append(label)
        cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 102, 204), 2)
        cv2.putText(img_rgb, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 102, 204), 2)

    try:
        faces = DeepFace.extract_faces(img_path=np.array(image), detector_backend="opencv", enforce_detection=False)
        for face in faces:
            fa = face["facial_area"]
            x = fa.get("x", 0)
            y = fa.get("y", 0)
            w = fa.get("w") or fa.get("width", 0)
            h = fa.get("h") or fa.get("height", 0)
            try:
                emotion_result = DeepFace.analyze(img_path=np.array(image), actions=["emotion"], enforce_detection=False)[0]
                emotion = emotion_result["dominant_emotion"]
            except:
                emotion = "Unknown"
            emotions.append(emotion)
            cv2.rectangle(img_rgb, (x, y), (x + w, y + h), (0, 204, 102), 2)
            cv2.putText(img_rgb, f"{emotion}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 204, 102), 2)
    except Exception as e:
        emotions.append("No face detected")

    annotated_img = Image.fromarray(img_rgb)
    object_summary = ", ".join(set(objects)) if objects else "None"
    emotion_summary = ", ".join(set(emotions)) if emotions else "None"

    return annotated_img, object_summary, emotion_summary


custom_css = """
.gradio-container {
  background: linear-gradient(to bottom right, #f0f8ff, #e6f7ff);
  font-family: 'Segoe UI', sans-serif;
  color: #333;
}
button {
  background-color: #00b8d4 !important;
  color: white !important;
  border-radius: 8px !important;
}
"""

with gr.Blocks(css=custom_css, title="Visual Analyzer") as app:
    gr.Markdown("""
    # 📊 Object and Emotion Visual Analyzer
    Upload an image to detect **objects** (YOLOv8) and **facial emotions** (DeepFace).
    Blue = Object box | Green = Face + Emotion box
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image")
            submit_btn = gr.Button("Analyze")
        with gr.Column():
            image_output = gr.Image(type="pil", label="Annotated Image")
            object_output = gr.Textbox(label="Detected Objects")
            emotion_output = gr.Textbox(label="Detected Emotions")

    submit_btn.click(analyze_image, inputs=image_input, outputs=[image_output, object_output, emotion_output])

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50829ce1bd27f36dd8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  Using cached deepface-0.0.93-py3-none-any.whl.metadata (30 kB)
  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached mtcnn-1.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached retina_face-0.0.17-py3-none-any.whl.metadata (10 kB)
  Using cached fire-0.7.0.tar.gz (87 kB)
  Preparing metadata (setup.py) ... done
  Using cached gunicorn-23.0.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached lz4-4.4.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
Using cached deepface-0.0.93-py3-none-any.whl (108 kB)
Using cached flask_cors-6.0.1-py3-none-any.whl (13 kB)
Using cached gunicorn-23.0.0-py3-none-any.whl (85 kB)
Using cached mtcnn-1.0.0-py3-none-any.whl (1.9 MB)
Using cached retina_face-0.0.17-py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.5 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=6ddb39fe93589fe9da838cd0be8edcd9264b0b78d610659b7363c0505

In [6]:
import gradio as gr
import cv2
from deepface import DeepFace
import numpy as np
from PIL import Image

def analyze_emotions(image):
    img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    emotions_summary = []

    try:
        faces = DeepFace.extract_faces(img_path=np.array(image), detector_backend="opencv", enforce_detection=False)

        for i, face in enumerate(faces):
            fa = face["facial_area"]
            x, y = fa.get("x", 0), fa.get("y", 0)
            w, h = fa.get("w", 0), fa.get("h", 0)

            try:
                result = DeepFace.analyze(img_path=np.array(image), actions=["emotion"], enforce_detection=False)[0]
                emotion = result["dominant_emotion"]
                confidence = result["emotion"][emotion]
                emotions_summary.append(f"Face {i+1}: {emotion} ({confidence:.2f}%)")
            except:
                emotion = "Unknown"
                emotions_summary.append(f"Face {i+1}: Unknown")

            cv2.rectangle(img_rgb, (x, y), (x + w, y + h), (0, 200, 100), 2)
            cv2.putText(img_rgb, emotion, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 100), 2)

    except:
        emotions_summary.append("⚠️ No faces detected.")

    final_image = Image.fromarray(img_rgb)
    return final_image, "\n".join(emotions_summary)


custom_css = """
.gradio-container {
  font-family: 'Segoe UI', sans-serif;
  background: #fafafa;
}

.main-page {
  display: flex;
  height: 100vh;
}

.sidebar {
  width: 220px;
  background-color: #ffffff;
  border-right: 1px solid #e0e0e0;
  padding: 30px 20px;
  height: 100vh;
}

.sidebar h2 {
  font-size: 22px;
  font-weight: 700;
  color: #333;
  margin-bottom: 20px;
}

.sidebar ul {
  list-style: none;
  padding: 0;
  font-size: 16px;
}

.sidebar li {
  margin: 12px 0;
  color: #555;
}

.content {
  flex: 1;
  padding: 40px 50px;
  overflow-y: auto;
}

h1.title {
  font-size: 28px;
  font-weight: 700;
  margin-bottom: 10px;
  color: #222;
}

h2.sub {
  font-size: 18px;
  color: #666;
  margin-bottom: 25px;
}

.upload-section, .result-section {
  background: #fff;
  padding: 25px;
  border-radius: 8px;
  box-shadow: 0 0 6px rgba(0,0,0,0.06);
  margin-bottom: 30px;
}

button {
  background-color: #007acc !important;
  color: white !important;
  font-weight: bold !important;
  border-radius: 6px !important;
  padding: 10px 20px !important;
}
"""

# --- Gradio App ---
with gr.Blocks(css=custom_css, title="Facial Emotion Recognition") as app:
    gr.HTML("""
    <div class='main-page'>
      <div class='sidebar'>
        <h2>Face Emotion</h2>
        <ul>
          <li>Overview</li>
          <li>Run</li>
          <li>Output</li>
          <li>About</li>
        </ul>
      </div>
      <div class='content'>
        <h1 class='title'>Facial Emotion Recognition</h1>
        <h2 class='sub'>AI-powered system to detect human emotions like Happy, Sad, Angry from images using Deep Learning.</h2>

        <div style="margin-top: 30px; font-size: 15px; color: #444; line-height: 1.7;">


        <h3>📌 Overview</h3>
        <p>
        Facial Emotion Recognition is a computer vision project that uses deep learning to identify the <b>emotional state</b> of people in images.
        By applying pre-trained models like <code>DeepFace</code>, we can analyze facial expressions and categorize them into emotions like <b>Happy</b>, <b>Sad</b>, <b>Angry</b>, <b>Surprise</b>, and more.
        </p>
        <p>
        This system has real-world applications in <b>mental health monitoring</b>, <b>education</b>, <b>marketing analytics</b>, and <b>human-computer interaction</b>.
        </p>


        <h3>🚀 Run Instructions</h3>
        <ul>
          <li>Upload a facial image (single or group)</li>
          <li>Click <b>"Analyze Emotions"</b></li>
          <li>The app detects each face and highlights its dominant emotion</li>
        </ul>
        <p>
        Green boxes indicate detected faces. Labels are drawn with the identified emotion (e.g., <code>Happy</code>, <code>Sad</code>).
        </p>


        <h3>🖼️ Output</h3>
        <p>
        After processing the image, the app shows:
        <ul>
          <li>✅ Annotated image with bounding boxes and emotion labels</li>
          <li>🧠 A summary of all detected emotions with confidence scores</li>
        </ul>
        </p>
        <p>Example Output:
        <pre>
Face 1: Happy (92.14%)
Face 2: Sad (66.87%)
        </pre>
        </p>

        <!-- ABOUT -->
        <h3>👨‍💻 About the Project</h3>
        <p>
        This project was built using:
        <ul>
          <li><b>DeepFace</b> - for face and emotion recognition</li>
          <li><b>OpenCV</b> - for image annotation and processing</li>
          <li><b>Gradio</b> - for creating a modern, web-based UI</li>
        </ul>

        </p>
        <p>
        This tool is ideal for students, researchers, and developers interested in AI-powered emotion classification. Future versions could integrate real-time webcam support, emotion graphs, or even log analysis.
        </p>

        <p style="margin-top: 25px; font-style: italic; font-size: 14px;">
        💡 Developed as a smart AI interface demo for interviews, workshops, and portfolio presentations.
        </p>

        </div>
    """)


    with gr.Column(elem_classes="upload-section"):
        image_input = gr.Image(type="pil", label="📤 Upload an Image")
        analyze_btn = gr.Button("🔍 Analyze Emotions")


    with gr.Column(elem_classes="result-section"):
        image_output = gr.Image(type="pil", label="📸 Annotated Output")
        emotion_output = gr.Textbox(label="📝 Detected Emotions", lines=8)


    analyze_btn.click(analyze_emotions, inputs=image_input, outputs=[image_output, emotion_output])

    gr.HTML("</div></div>")

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f204a7fe62abcd7d3e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
!pip install gradio
!pip install deepface
!pip install ultralytics
!pip install gradio ultralytics deepface opencv-python matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.1 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=fb3adee0ad9352612d60c3900b47893b257466fe78cb7115e140a3a363b42d85
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6